In [1]:
from pathlib import Path
import numpy as np
import h5py


def _read_arr_and_attrs(h5obj, key):
    ds = h5obj[key]
    return ds[...], dict(ds.attrs)


def read_hydro_profiles_hdf5(
    path,
    component=None,                 # e.g. "DM", "Gas", "Stars", "All" (or None for all)
    load_cy=True,
    expand_rbins=False,             # if True: repeat (n_bins,) -> (n_halo,n_bins)
    expand_mass=False,              # if True: repeat (n_halo,) -> (n_halo,n_bins)
):
    """
    Read the HYDRO file only written by save_hydro_profiles_hdf5().

    Returns:
      - if component is None:
          dict with keys: meta, r_bins, halo_id, halo_mass, lensing{comp}, (optional) cy
      - if component is not None:
          dict mimicking the old read_lensing_h5 style:
            r_bins, Sigma, DeltaSigma, halo_mass, halo_id, (optional) cy_at_rp
    Each array comes with a parallel 'units' dict.
    """
    path = Path(path)

    with h5py.File(path, "r") as f:
        meta = dict(f.attrs)

        r_bins, r_bins_attr = _read_arr_and_attrs(f, "bins/r_bins")
        halo_id, halo_id_attr = _read_arr_and_attrs(f, "halos/halo_id")
        halo_mass, halo_mass_attr = _read_arr_and_attrs(f, "halos/halo_mass")

        n_halo = halo_id.shape[0]
        n_bins = r_bins.shape[0]

        if expand_rbins:
            r_bins2d = np.repeat(r_bins[None, :], n_halo, axis=0)
        else:
            r_bins2d = r_bins

        if expand_mass:
            halo_mass2d = np.repeat(halo_mass[:, None], n_bins, axis=1)
        else:
            halo_mass2d = halo_mass

        # helper to read one component
        def _read_comp(comp):
            Sig, Sig_attr = _read_arr_and_attrs(f, f"lensing/{comp}/Sigma")
            DSig, DSig_attr = _read_arr_and_attrs(f, f"lensing/{comp}/DeltaSigma")
            return {
                "Sigma": Sig,
                "DeltaSigma": DSig,
                "units": {
                    "Sigma": Sig_attr.get("units"),
                    "DeltaSigma": DSig_attr.get("units"),
                },
            }

        # optional cy
        cy = None
        cy_units = None
        if load_cy and "compton_y/cy_at_rp" in f:
            cy, cy_attr = _read_arr_and_attrs(f, "compton_y/cy_at_rp")
            cy_units = cy_attr.get("units")

        # --- return all components ---
        if component is None:
            comps = {}
            for comp in f["lensing"].keys():
                comps[comp] = _read_comp(comp)

            out = {
                "meta": meta,
                "r_bins": r_bins2d,
                "halo_id": halo_id,
                "halo_mass": halo_mass2d,
                "lensing": comps,
                "units": {
                    "r_bins": r_bins_attr.get("units"),
                    "halo_id": halo_id_attr.get("units"),
                    "halo_mass": halo_mass_attr.get("units"),
                },
            }
            if cy is not None:
                out["cy_at_rp"] = cy
                out["units"]["cy_at_rp"] = cy_units
            return out

        # --- return a single component (old-style dict) ---
        comp = str(component)
        comp_dat = _read_comp(comp)

        out = {
            "meta": meta,
            "r_bins": r_bins2d,
            "Sigma": comp_dat["Sigma"],
            "DeltaSigma": comp_dat["DeltaSigma"],
            "halo_mass": halo_mass2d,
            "halo_id": halo_id,
            "units": {
                "r_bins": r_bins_attr.get("units"),
                "Sigma": comp_dat["units"]["Sigma"],
                "DeltaSigma": comp_dat["units"]["DeltaSigma"],
                "halo_mass": halo_mass_attr.get("units"),
                "halo_id": halo_id_attr.get("units"),
            },
        }
        if cy is not None:
            out["cy_at_rp"] = cy
            out["units"]["cy_at_rp"] = cy_units
        return out


def read_dmo_lensing_hdf5(
    path,
    expand_rbins=False,   # repeat (n_bins,) -> (n_halo,n_bins)
    expand_mass=False,    # repeat (n_halo,) -> (n_halo,n_bins)
):
    """
    Read the DMO file written by save_dmo_lensing_hdf5().
    Returns a dict similar to your old read_lensing_h5 style.
    """
    path = Path(path)

    with h5py.File(path, "r") as f:
        meta = dict(f.attrs)

        r_bins, r_bins_attr = _read_arr_and_attrs(f, "bins/r_bins")
        halo_mass, halo_mass_attr = _read_arr_and_attrs(f, "halos/halo_mass")
        Sigma, Sigma_attr = _read_arr_and_attrs(f, "lensing/Sigma")
        DeltaSigma, DeltaSigma_attr = _read_arr_and_attrs(f, "lensing/DeltaSigma")

        n_halo = Sigma.shape[0]
        n_bins = r_bins.shape[0]

        if expand_rbins:
            r_bins = np.repeat(r_bins[None, :], n_halo, axis=0)
        if expand_mass:
            halo_mass = np.repeat(halo_mass[:, None], n_bins, axis=1)

        return {
            "meta": meta,
            "r_bins": r_bins,
            "Sigma": Sigma,
            "DeltaSigma": DeltaSigma,
            "halo_mass": halo_mass,
            "units": {
                "r_bins": r_bins_attr.get("units"),
                "Sigma": Sigma_attr.get("units"),
                "DeltaSigma": DeltaSigma_attr.get("units"),
                "halo_mass": halo_mass_attr.get("units"),
            },
        }


In [2]:
# --- HYDRO: read DM only (like old style) ---

proj_depth = 300 # units: cMpc/h

hydro_file = f"hydro_lensing_cy_projdepth{proj_depth}cMpch.hdf5"
dm = read_hydro_profiles_hdf5(hydro_file, component="DM", load_cy=True, expand_rbins=False)

rp = dm["r_bins"]                 # (20,) by default
Sigma_dm = dm["Sigma"]            # (83731, 20)
DS_dm = dm["DeltaSigma"]          # (83731, 20)
halo_mass_hydro = dm["halo_mass"] # (83731,)
halo_id = dm["halo_id"]           # (83731,)
cy = dm.get("cy_at_rp", None)     # (83731, 20) if present

print("Units:", dm["units"])
print("Meta:", dm["meta"])

# --- HYDRO: read everything at once (nested dict) ---
hyd = read_hydro_profiles_hdf5(hydro_file, component=None, load_cy=True)
Sigma_stars = hyd["lensing"]["Stars"]["Sigma"]
DS_all = hyd["lensing"]["All"]["DeltaSigma"]

# --- DMO ---
dmo_file = f"dmo_lensing_projdepth{proj_depth}cMpch.hdf5" 
dmo = read_dmo_lensing_hdf5(dmo_file)

rp_dmo = dmo["r_bins"]                # (20,)
Sigma_dmo = dmo["Sigma"]              # (102511, 20)
DS_dmo = dmo["DeltaSigma"]            # (102511, 20)
halo_mass_dmo = dmo["halo_mass"]      # (102511,)
print("Units:", dmo["units"])
print("Meta:", dmo["meta"])



Units: {'r_bins': 'cMpc/h', 'Sigma': 'Msun*h/(cMpc)^2', 'DeltaSigma': 'Msun*h/(cMpc)^2', 'halo_mass': 'Msun/h', 'halo_id': 'dimensionless', 'cy_at_rp': 'dimensionless'}
Meta: {'file_format': 'flamingo_hydro_profiles_v1', 'h': 0.6809999999999999, 'n_bins': 20, 'n_halo': 83731, 'proj_depth_cMpc_over_h': 300.0, 'redshift': 0.3, 'scale_factor': 0.7692307692307692, 'sim': 'FLAMINGO L1000N3600 HYDRO_FIDUCIAL', 'snap': '0072'}
Units: {'r_bins': 'cMpc/h', 'Sigma': 'Msun*h/(cMpc)^2', 'DeltaSigma': 'Msun*h/(cMpc)^2', 'halo_mass': 'Msun/h'}
Meta: {'file_format': 'flamingo_dmo_lensing_v1', 'h': 0.6809999999999999, 'n_bins': 20, 'n_halo': 102511, 'proj_depth_cMpc_over_h': 300.0, 'redshift': 0.3, 'scale_factor': 0.7692307692307692, 'sim': 'FLAMINGO L1000N3600 DMO_FIDUCIAL', 'snap': '0072'}


In [ ]:
# ### r_p bins used for lensing/cy calculation and for both hydro and DMO simulations

# r_min, r_max, n_bins = 0.1, 30.0, 20
# R_comv_bins = np.logspace(np.log10(r_min), np.log10(r_max), n_bins+1)
# R_comv_bins_inner = np.concatenate([[0.0], R_comv_bins])
# rpmid_list = np.sqrt(R_comv_bins_inner[:-1] * R_comv_bins_inner[1:])[1:] This is rp bin centers for lensing/cy profiles
